# Pairs Trading and Market Making Algorithm

## Aim
The aim of this algorithm is to identify and trade temporary price differences between correlated securities listed on the optibook exchange. The strategy compares the best bid and ask prices of ASML and ASML_DUAL to detect and exploit price discrepancies, where one instrument can be bought at a lower price and the other sold at a higher price. It also includes a market making strategy which identifies opportunities to place bids and offers with a spread in the order book, such that other market participates will lift the offer and hit the bid.

The algorithm also uses a stellage to adjust its position according to the size of the spread. This allows the algorithm to increase its position as the spread increase which ensures that the program doesn't use all of its position on a small spread. When the spread narrows, the algorithm gradually reduces its position, allowing it to realise profits while avoiding of holding a large position after the pricing difference disappears.

Market markers play an important role in financial markerts by continuously providing bid and ask prices at which other participants can buy or sell securities. This improves liquidity, reduces the time required to execute trades and helps narrow bid-ask spreads. Market makers earn revenue primarily from the difference between their buying and selling prices, but they also fdace invenotry risk when trades leave them holding an uneven position.

## Outline of Strategy

The final algorithm combines two distinct trading strategies for ASML and ASML_DUAL: a passive market-making strategy that continuously quotes both instruments, and an active spread-arbitrage strategy that trades on temporary price differences between them. Rather than running simultaneously, the two strategies are combined by alternating between them in fixed-length phases, so that at any given moment only one strategy is actively managing risk in the market.

#### Market-Making Component

During its active phase, the market-making strategy retrieves the best bid and ask price for each instrument and calculates a mid price. It then adjusts this mid price using an inventory skew based on the algorithm's current position in that instrument: as the position grows in either direction, the skew shifts both the bid and ask quotes to make it relatively more attractive for the market to trade in the direction that returns the position toward flat, and relatively less attractive to trade in the direction that would increase it further. The size of this shift is proportional to how large a fraction of the position limit is currently in use, so that quotes near a flat position are only lightly adjusted, while quotes near the position limit are adjusted more strongly.

Around this skewed mid price, the algorithm places a bid and an ask separated by a fixed quote width. The size offered on each side is not fixed, but is instead scaled according to the relative depth of resting volume on the bid and ask sides of the visible order book. If the book shows more resting volume on the bid side than the ask side, the algorithm increases the size it is willing to sell and reduces the size it is willing to buy, and vice versa. This is intended to make the algorithm's own quoted sizes responsive to the prevailing supply and demand imbalance in the market, rather than always quoting a
constant volume regardless of conditions. Before every quote is placed, the algorithm checks that filling it in full would not push the position beyond the exchange's position limit, and quotes are only submitted on the side(s) where this is guaranteed to hold.

#### Arbitrage Component

During its active phase, the arbitrage strategy continuously retrieves the best bid and ask prices for both instruments and evaluates two possible trading opportunities. A short-spread opportunity occurs when the best bid price of ASML is greater than the best ask price of ASML_DUAL, in which case the algorithm sells ASML and buys ASML_DUAL. A long-spread opportunity occurs when the best bid price of ASML_DUAL is greater than the best ask price of ASML, in which case the algorithm buys ASML and sells ASML_DUAL. The underlying idea is that because the two instruments represent the same underlying company, any such difference should be temporary and should narrow over time; by holding opposite positions in the two instruments, the algorithm aims to profit from that convergence while remaining largely neutral to the direction of the underlying share price itself.

A key feature of this strategy is the use of a stellage to size the position according to the magnitude and direction of the observed spread. The current spread between the two instruments is divided by an observed maximum spread to produce a normalised value between approximately −1 and 1, where positive values favour selling ASML and buying ASML_DUAL, and negative values favour the opposite. The maximum spread used for this normalisation was set at 1.5, chosen with a margin above the largest spread observed during testing (approximately 1.1), to avoid saturating the normalised measure under typical conditions.

The stellage itself maps ranges of the normalised spread to acceptable position ranges, with small spreads corresponding to modest position ranges and progressively larger spreads corresponding to substantially larger ones, up to a position of 100 lots at the most extreme observed levels. This structure was chosen after observing that small differences between ASML and ASML_DUAL occurred considerably more often than large ones: by using tighter position bands near zero spread, the algorithm enters positions gradually as small opportunities appear, and only commits larger size once a more significant spread is observed. The same structure allows the position to be reduced in the same graduated way as a spread narrows, since the acceptable range itself shrinks and the algorithm trades only the amount needed to bring its position back within the new, smaller range. Using acceptable ranges rather than exact target positions also avoids unnecessary trading: if the existing position already lies inside the current range, no further order is submitted.

#### Combining the Two Strategies

Running both strategies continuously and simultaneously against the same two instruments was considered but rejected, primarily because it would make risk difficult to reason about: each strategy would be reacting to a position that the other strategy was also actively changing, and a single shared position limit would need to be split between them in a way that could easily leave one strategy unable to act when it most needed to.

Instead, the two strategies are combined through explicit phase alternation. The algorithm runs a fixed number of loop iterations of active spread-arbitrage trading, then cancels all outstanding orders on both instruments, then runs the same fixed number of iterations of passive market-making quoting, cancels all outstanding orders again, and repeats. At any point in time only one strategy is placing new orders, and every transition between strategies begins from a clean order book with no orders left over from the previous phase. Both strategies size their orders directly from the exchange's
live reported position rather than from any locally maintained estimate of position, so that whichever strategy is active always has an accurate, up-to-date view of the position the other strategy most recently left it, including the position generated during the market-making phase immediately before an arbitrage phase begins, and vice versa.

A shared position-balancing routine also runs during the active trading phase, independently of the arbitrage logic's own sizing. Because the arbitrage strategy submits its two legs as separate IOC orders, one leg may fill while the other does not, leaving an unwanted outright position in a single instrument rather than a balanced spread position. The position-balancing function checks the combined position across both instruments and, if it is not close to zero, submits a small corrective order into whichever instrument currently offers the more favourable price, gradually pulling any such spill back toward balance without waiting for the next spread opportunity to do so.

#### Given More Time

Given more time, several aspects of the combined strategy could be improved. A larger historical sample of spread observations would allow the maximum spread and the stellage boundaries to be chosen statistically rather than primarily through visual inspection. The market-making component could incorporate available volume at multiple levels of the order book, rather than only the best bid and ask, when deciding how competitively to quote. Finally, the fixed-length phase alternation itself is a relatively coarse way of sharing time between the two strategies; a more adaptive approach, where the algorithm spends more time in whichever mode currently offers a clearer edge, could potentially make better use of the trading hour than a rigid, evenly-split schedule.

## Results
#### Profit / Loss
The algorithm generated a positive profit of approximately $660 over the one-hour evaluation period, with cumulative PnL increasing from approximately $12,730 to $13,390 (Figure 1). Rather than exhibiting smooth linear growth, the equity curve displays a staircase-like pattern consisting of periods of relatively stable performance followed by discrete increases in profitability. This behaviour is consistent with the underlying design of the strategy. Profits are realised only when arbitrage opportunities emerge or when passive market-making orders are executed, meaning returns naturally occur in bursts rather than continuously.

Although the equity curve contains several short drawdowns, none were sufficiently large to offset the overall upward trend. This suggests that inventory management and spill-position balancing were generally effective at limiting prolonged directional exposure despite operating close to the exchange's inventory limits. The largest increase in cumulative profit occurred during a relatively short interval, indicating that favourable market conditions allowed multiple arbitrage opportunities to be captured within a brief period. Overall, the positive and relatively stable growth in cumulative PnL demonstrates that the strategy remained profitable throughout the evaluation period while operating continuously for the full one-hour trading session.

![image (1).png](image%20%281%29.png)

#### Turnover
The algorithm executed a substantial trading volume throughout the evaluation period. ASML recorded approximately 58,800 filled lots, while ASML_DUAL recorded approximately 29,600 filled lots, resulting in a combined turnover of approximately 88,400 lots (Figure 2). Importantly, the bid and ask volumes for both instruments were closely balanced, indicating that the strategy actively participated on both sides of the market rather than accumulating a persistent directional position.

High turnover alone is not necessarily indicative of a successful trading strategy, as excessive trading can simply increase transaction costs and inventory risk. However, in this case the high turnover was accompanied by positive profitability, suggesting that the algorithm was successfully converting trading opportunities into realised gains rather than merely generating unnecessary order flow. Furthermore, the balanced nature of the executed trades demonstrates that the algorithm fulfilled one of the primary objectives of market making by consistently providing liquidity to other market participants.
![image copy 2.png](image%20copy%202.png)

#### Realized spread / edge
The realised execution prices reveal that the two components of the strategy contributed differently to overall profitability. For ASML, the average selling price exceeded the average purchase price by approximately $0.038 per unit, indicating that the passive market-making strategy was consistently capturing a positive bid-ask spread. This demonstrates that the quoted prices were sufficiently competitive to generate spread revenue while maintaining acceptable inventory levels.

Conversely, ASML_DUAL exhibited a negative realised spread of approximately -$0.048 per unit, implying that passive quoting on this instrument was not independently profitable. Despite this, the overall trading strategy remained profitable throughout the evaluation period. This suggests that the arbitrage component successfully offset the weaker passive performance by exploiting temporary pricing discrepancies between the two order books. Consequently, the results indicate that overall profitability was driven by the interaction between passive market making and active arbitrage rather than by either strategy operating independently.

![image copy 3.png](image%20copy%203.png)

# 
#### Position management / risk
Inventory management represents one of the principal risks associated with any market-making strategy, as sustained directional exposure can quickly erode accumulated profits. Throughout the evaluation period, inventory levels frequently approached the exchange's maximum allowable position of ±100 lots, with several occasions where positions briefly reached within a small margin of this limit as shown in the image below. Although this demonstrates that the algorithm actively utilised available inventory capacity to maximise trading opportunities, it also indicates that risk exposure occasionally became elevated.

Despite operating close to these limits, the algorithm successfully completed the one-hour trading session without breaching exchange constraints. This represents a significant improvement over earlier versions of the algorithm, where inconsistent inventory tracking resulted in pre-trade position-limit violations. The combination of inventory skew, spill-position balancing and direct querying of exchange positions proved effective in maintaining compliance while allowing the strategy to remain sufficiently aggressive to capture profitable trading opportunities. Nevertheless, the frequency with which inventory approached the maximum position limit suggests that further refinement of the skew function or earlier rebalancing could reduce tail risk without materially reducing profitability.
![image copy.png](image%20copy.png)

#### Summary
Overall, the algorithm successfully achieved its primary objective of generating profit while providing continuous liquidity to the Optibook exchange. During the one-hour evaluation period, it generated approximately $660 in profit while executing approximately 88,400 lots of trading volume across both instruments. The results demonstrate that passive market making and active statistical arbitrage complemented one another effectively: market making generated consistent spread revenue on ASML, while arbitrage compensated for weaker passive performance on ASML_DUAL by exploiting temporary pricing inefficiencies.
The strategy also demonstrated a high degree of operational robustness. It completed the full evaluation period without interruption, maintained balanced trading activity on both sides of the market and remained within the exchange's inventory constraints. While opportunities remain to further refine inventory management and parameter selection, the overall performance indicates that the proposed strategy provides a meaningful improvement over the baseline implementation and successfully balances profitability, liquidity provision and risk management.


## Evolution of Algorithm

#### 1. Two strategies, built and tested separately

The market-making and arbitrage components were originally written and tested as two
independent scripts. The market-making script quoted both instruments around a
skewed mid price, with the skew and quote size scaled to inventory and order-book
depth. The arbitrage script traded crossed order books between ASML and ASML_DUAL,
sizing its position using a stellage based on the normalised spread between the two.
Each read positions and managed its own single limit, and each behaved correctly on
its own.

#### 2. Combining them: phase alternation

Running both simultaneously against the same instruments was rejected, since neither
would have visibility into what the other was doing. Instead, the two were combined by
alternating fixed-length phases, a set number of loops of active arbitrage trading,
then the same number of loops of passive quoting, cancelling all resting orders at
every switch. `LOOPS_PER_PHASE` was tuned from 10, to 30, to a final value of 100 after
short phases proved to spend too much time on cancel/restart overhead relative to
trading.

#### 3. A bug that only appeared once combined

Carried over from standalone development, the market-making logic computed its own
exposure as `mm_position = total_position - arb_position`, using a locally tracked
`arb_position` counter that the arbitrage side updated optimistically after sending
orders. This never mattered when each script ran alone. Once combined, the
market-making phase's own fills changed the real position without ever updating that
counter, so the two components' views of "current position" drifted apart. The fix was
to have both components query `exchange.get_positions()` directly before every sizing
decision, and to collapse the two separate limits into one shared limit.

#### 4. A real position-limit breach

The same class of bug resurfaced on the arbitrage side and caused a live disconnect:
`Pre-trade position limit (-100) breached on instrument ASML_DUAL. Old position=-70
New worst-case position=-145`. The arbitrage phase was still sizing off its own
tracked position rather than the real one, so it authorised a trade that pushed the
true position 45 lots past the limit. `trade_loop` was rewritten to size directly from
`exchange.get_positions()`, with a small safety margin below the nominal limit.

#### 5. Adding measurement, then using it

After the disconnect, structured CSV logging was added, one row per loop per
instrument (position, PnL, quotes) and one row per fill, written incrementally so a
crash wouldn't lose data. The first full hour of logged data looked healthy on
PnL alone (+$11,700), but turnover was 84,777 units for a net change of only -$19.96
(PnL-per-turnover of -0.0002). The data showed why: the quoted spread (0.20) was
significantly wider than the market's average (0.12), quotes sat behind the touch on
both sides, and PnL volatility rose with position size (bucketed analysis: -0.28
average PnL change in the largest-position bucket vs. slightly positive in the
smallest). This led to two targeted changes: tightening `quote_spread` toward the
market average, and increasing `max_skew` so it could meaningfully pull the book back
toward flat near the position limit.

#### 6. A methodological lesson

An attempt to eyeball a "before/after" comparison from the live PnL chart wasn't
reliable, since the market moves independently of any code change and the windows
weren't controlled for duration or starting position. The timestamped, phase-labelled
CSV logs are used instead for the final comparison in this report.

#### 7. Summary

**Worked:** validating each strategy independently before combining them, so bugs from
combination could be isolated from pre-existing ones; phase alternation with a full
order cancellation between switches; collapsing to one shared, freshly-queried
position source; logging before tuning; hypothesis-driven analysis of the logged data
rather than judging by headline PnL alone.

**Didn't work:** carrying each script's own position tracking into the combined bot
unreconciled; treating an order sent as equivalent to a confirmed fill; and reading a
live dashboard chart as evidence for a parameter change's effect.

## Algorithm

In [ ]:
import time
import logging
from optibook.synchronous_client import Exchange
import datetime
import math
import csv
import os
import math
logger = logging.getLogger('client')
logger.setLevel('ERROR')

print("Setup was successful.")

logging.getLogger('client').setLevel('ERROR')



LOOPS_PER_PHASE = 100      # how many loop iterations each phase runs before switching

# How long to run in total before stopping cleanly and saving the logs.
RUN_DURATION_SECONDS = 3600   # 1 hour


# # # # # # # # # # # # # # # # # # # # # # # # # # # # # #
# Logging                                                    #
# Writes two CSVs for post-run analysis:                     #
#   loop_log_*.csv  - one row per instrument per loop:        #
#                     position, PnL, market state, quotes     #
#   trade_log_*.csv - one row per individual fill              #
# # # # # # # # # # # # # # # # # # # # # # # # # # # # # #

LOG_DIR = 'run_logs'
_run_label = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
LOOP_LOG_PATH = os.path.join(LOG_DIR, f'Trading_Logg_loop_{_run_label}.csv')
TRADE_LOG_PATH = os.path.join(LOG_DIR, f'Trading_Logg_trades_{_run_label}.csv')

_LOOP_FIELDS = [
    'timestamp', 'phase', 'instrument',
    'position', 'pnl',
    'best_bid', 'best_ask', 'mid',
    'quote_bid_price', 'quote_bid_vol',
    'quote_ask_price', 'quote_ask_vol',
]
_TRADE_FIELDS = [
    'timestamp', 'phase', 'instrument',
    'side', 'price', 'volume',
]

_loop_file = None
_loop_writer = None
_trade_file = None
_trade_writer = None
_seen_trade_ids = set()


def init_logging():
    """Call once at the start of main() before the trading loop begins."""
    global _loop_file, _loop_writer, _trade_file, _trade_writer

    os.makedirs(LOG_DIR, exist_ok=True)

    _loop_file = open(LOOP_LOG_PATH, 'w', newline='')
    _loop_writer = csv.DictWriter(_loop_file, fieldnames=_LOOP_FIELDS)
    _loop_writer.writeheader()

    _trade_file = open(TRADE_LOG_PATH, 'w', newline='')
    _trade_writer = csv.DictWriter(_trade_file, fieldnames=_TRADE_FIELDS)
    _trade_writer.writeheader()

    print(f"Logging to:\n  {LOOP_LOG_PATH}\n  {TRADE_LOG_PATH}")


def safe_get_pnl(exchange):
    """get_pnl() shape can vary slightly by course build - adjust here if needed."""
    try:
        return exchange.get_pnl()
    except Exception as e:
        print(f"[logger] get_pnl failed: {e}")
        return None


def log_loop_state(exchange, phase, instrument_id, best_bid, best_ask,
                    quote_bid_price=None, quote_bid_vol=None,
                    quote_ask_price=None, quote_ask_vol=None):
    """Log one instrument's state for this loop iteration: position, PnL,
    market prices, and (if MM phase) the quotes we posted."""
    positions = exchange.get_positions()
    position = positions.get(instrument_id, 0)
    pnl = safe_get_pnl(exchange)
    mid = (best_bid + best_ask) / 2 if (best_bid is not None and best_ask is not None) else None

    row = {
        'timestamp': datetime.datetime.now().isoformat(),
        'phase': phase,
        'instrument': instrument_id,
        'position': position,
        'pnl': pnl,
        'best_bid': best_bid,
        'best_ask': best_ask,
        'mid': mid,
        'quote_bid_price': quote_bid_price,
        'quote_bid_vol': quote_bid_vol,
        'quote_ask_price': quote_ask_price,
        'quote_ask_vol': quote_ask_vol,
    }
    _loop_writer.writerow(row)
    _loop_file.flush()


def log_new_trades(exchange, instrument_id, phase):
    """Poll and log any new fills for one instrument since the last call,
    for turnover tracking. Defensive about attribute names since the exact
    Trade object schema can vary by course build - if nothing gets logged
    despite fills happening, print(vars(trade)) on one and adjust below."""
    try:
        new_trades = exchange.poll_new_trades(instrument_id)
    except Exception as e:
        print(f"[logger] poll_new_trades failed for {instrument_id}: {e}")
        return

    if not new_trades:
        return

    for trade in new_trades:
        price = getattr(trade, 'price', None)
        volume = getattr(trade, 'volume', getattr(trade, 'quantity', None))
        side = getattr(trade, 'side', getattr(trade, 'aggressor_side', None))
        trade_id = getattr(trade, 'order_id', None) or getattr(trade, 'trade_id', None) or id(trade)

        if trade_id in _seen_trade_ids:
            continue
        _seen_trade_ids.add(trade_id)

        if price is None or volume is None:
            print(f"[logger] Unrecognized trade object fields: "
                  f"{vars(trade) if hasattr(trade, '__dict__') else trade}")

        row = {
            'timestamp': datetime.datetime.now().isoformat(),
            'phase': phase,
            'instrument': instrument_id,
            'side': side,
            'price': price,
            'volume': volume,
        }
        _trade_writer.writerow(row)
    _trade_file.flush()


def close_logging():
    """Call once when the bot shuts down (e.g. in a finally block)."""
    if _loop_file:
        _loop_file.close()
    if _trade_file:
        _trade_file.close()
    print(f"Logs closed:\n  {LOOP_LOG_PATH}\n  {TRADE_LOG_PATH}")





# Market Making Algorithim

INSTRUMENT_IDS = ['ASML', 'ASML_DUAL']

quote_spread = 0.20     # total width of our quote around mid (e.g. 0.10 each side)
position_limit = 100    # max absolute position allowed by the exchange, per instrument
sleep_seconds = 1       # how often to refresh quotes


# --- Inventory skew based on current position ---
max_skew = 0.06         # max price shift (in $) applied when at full position limit


def get_top_of_book(exchange, instrument_id):
    book = exchange.get_last_price_book(instrument_id)
    if not book or not book.bids or not book.asks:
        return None, None
    return book.bids[0].price, book.asks[0].price


def get_quote_volume(total_volume_bid,total_volume_ask):
    # Scale quote size relative to the market's postion i.e if more bids then place larger volume under bids and vice versa
    if total_volume_bid == 0 or total_volume_ask == 0:
        bid_volume = 0
        ask_volume = 0
    else:
        bid_volume = int(round(total_volume_ask/(total_volume_ask+total_volume_bid)*position_limit))
        ask_volume = int(round(total_volume_bid/(total_volume_ask+total_volume_bid)*position_limit))
    return ask_volume, bid_volume
    

def get_skew(current_position):
    # Shift both quotes based on current position, so the bot leans back toward flat instead of drifting further from it.


    # Fraction of the position limit currently used, from -1.0 to 1.0
    fraction = current_position / position_limit
    return -fraction * max_skew


def quote_instrument(exchange, instrument_id):
    # Cancel old quotes and place fresh bid/ask quotes for one instrument.
    # 1. Cancel all existing orders for this instrument before re-quoting
    exchange.delete_orders(instrument_id)
    book = exchange.get_last_price_book(instrument_id)

    # 2. Get current best bid/ask and derive mid price + market spread
    best_bid, best_ask = get_top_of_book(exchange, instrument_id)
    if best_bid is None or best_ask is None:
        print(f"[{instrument_id}] No valid order book yet, skipping this cycle.")
        return

    mid_price = (best_bid + best_ask) / 2

    # 3. Get current position first, since it affects our pricing
    positions = exchange.get_positions()
    current_position = positions.get(instrument_id, 0)

    # 4. Compute inventory skew and apply it to a skewed mid price
    skew = get_skew(current_position)
    skewed_mid = mid_price + skew

    # 5. Compute our bid/ask quotes around the skewed mid
    half_spread = quote_spread / 2
    bid_price = round(skewed_mid - half_spread, 2)
    ask_price = round(skewed_mid + half_spread, 2)

    # 6. Compute quote size based on how wide the market positions are
    if not book or not book.bids:
        total_volume_bid = 0
    else:
        total_volume_bid = sum(level.volume for level in book.bids)
    if not book or not book.asks:
        total_volume_ask = 0
    else:
        total_volume_ask = sum(level.volume for level in book.asks)
    quote_volume_ask, quote_volume_bid = get_quote_volume(total_volume_bid,total_volume_ask)

    # 7. Place bid only if buying more wouldn't breach the long limit
    if quote_volume_bid > 0 and current_position + quote_volume_bid <= position_limit:
        exchange.insert_order(
            instrument_id,
            price=bid_price,
            volume=quote_volume_bid,
            side='bid',
            order_type='limit',
        )

    # 8. Place ask only if selling more wouldn't breach the short limit
    if quote_volume_ask > 0 and current_position - quote_volume_ask >= -position_limit:
        exchange.insert_order(
            instrument_id,
            price=ask_price,
            volume=quote_volume_ask,
            side='ask',
            order_type='limit',
        )

    print(f"[{instrument_id}] Mid: {mid_price:.2f} | Skew: {skew:+.3f} | "
      f"BidVol: {quote_volume_bid:.1f} | AskVol: {quote_volume_ask:.1f} | "
      f"Bid: {bid_price:.2f} | Ask: {ask_price:.2f} | Position: {current_position}")


# Arbitrage Algorithim

# # # # # # # # # # # # # # # # # # # # # # # # # # # # # #
#  This algorithm actively trades on crossed orderbooks.  #
# # # # # # # # # # # # # # # # # # # # # # # # # # # # # #



# # # # # # # #
# Parameters  #
# # # # # # # #

# Names of the two dual stocks I want to trade against each other.
stock_A = 'ASML'
stock_B = 'ASML_DUAL'

# How much volume will I shoot for on an opportunity:
opp_volume = 25
position_limit = 100
position_buffer = 10
trading_limit = position_limit - position_buffer
spread_max = 1.5

# Some parameters for my position balancing function:
max_tolerance_to_rebalance = 0.3
volume_to_balance = 2

# # # # # # #
# Functions #
# # # # # # #
def positions(exchange):
    while True:
        positions_initial = exchange.get_positions()
        current_position_A = positions_initial.get(stock_A,0)
        current_position_B = positions_initial.get(stock_B,0)

        order_book_A = exchange.get_last_price_book(stock_A)
        order_book_B = exchange.get_last_price_book(stock_B)

        books_are_valid = (
            order_book_A is not None
            and order_book_B is not None
            and order_book_A.bids
            and order_book_A.asks
            and order_book_B.bids
            and order_book_B.asks
        )

        if books_are_valid:
            best_bid_A = order_book_A.bids[0].price
            best_ask_A = order_book_A.asks[0].price

            best_bid_B = order_book_B.bids[0].price
            best_ask_B = order_book_B.asks[0].price

            return (
                positions_initial,
                current_position_A,
                current_position_B,
                order_book_A,
                order_book_B,
                best_bid_A,
                best_ask_A,
                best_bid_B,
                best_ask_B,
            )

        time.sleep(0.1)

def stellage(current_position_A, best_bid_A, best_ask_A, best_bid_B, best_ask_B):
    short_spread = (best_bid_A - best_ask_B) / spread_max
    long_spread = (best_bid_B - best_ask_A) / spread_max

    P = [
        [1.00, -100, -100], [0.5, -75, -100], [0.2, -50, -75], [0.1, -35, -50],[0.05,-20,-35], [0, 0, 0], [-0.05,20,35],
        [-0.1, 35, 50], [-0.2, 50, 75], [-0.5, 75, 100], [-1, 100, 100]
    ]

    if short_spread > 0 and short_spread >= long_spread:
        spread_level = short_spread
    elif long_spread > 0:
        spread_level = -long_spread
    else:
        spread_level = 0

    value, range_a, range_b = min(P, key=lambda row: abs(row[0] - spread_level))
    lower_bound = min(range_a, range_b)
    upper_bound = max(range_a, range_b)

    if current_position_A < lower_bound:
        status = "below"                                   # need to BUY A (long spread direction)
        target_volume = int(lower_bound - current_position_A)
    elif current_position_A > upper_bound:
        status = "above"                                    # need to SELL A (short spread direction)
        target_volume = int(current_position_A - upper_bound)
    else:
        status = "within"
        target_volume = 0

    return status, target_volume

    

# Searches for opportunities and attempts to trade
def trade_loop(exchange, positions_initial, target_status, target_volume, best_bid_A, best_ask_A, best_bid_B, best_ask_B):
    long_spread_opp_detected = False
    short_spread_opp_detected = False

    if best_bid_A > best_ask_B:
        short_spread_opp_detected = True
        print('OPPORTUNITY DETECTED: Sell A AND Buy B')

        if target_status == "above" and target_volume > 0:
            room_A = positions_initial.get(stock_A,0) + position_limit   # room to sell A further
            room_B = position_limit - positions_initial.get(stock_B,0)   # room to buy B further
            trade_volume = min(target_volume, room_A, room_B)
            if trade_volume > 0:
                exchange.insert_order(stock_A, price=best_bid_A, volume=trade_volume, side='ask', order_type='ioc')
                exchange.insert_order(stock_B, price=best_ask_B, volume=trade_volume, side='bid', order_type='ioc')
            else:
                print('No room to trade.')
        else:
            print('Opportunity detected, but position already within target band.')

    elif best_bid_B > best_ask_A:
        long_spread_opp_detected = True
        print('OPPORTUNITY DETECTED: Sell B AND Buy A')

        if target_status == "below" and target_volume > 0:
            room_A = position_limit - positions_initial.get(stock_A,0)   # room to buy A further
            room_B = positions_initial.get(stock_B,0) + position_limit   # room to sell B further
            trade_volume = min(target_volume, room_A, room_B)
            if trade_volume > 0:
                exchange.insert_order(stock_B, price=best_bid_B, volume=trade_volume, side='ask', order_type='ioc')
                exchange.insert_order(stock_A, price=best_ask_A, volume=trade_volume, side='bid', order_type='ioc')
            else:
                print('No room to trade.')
        else:
            print('Opportunity detected, but position already within target band.')
    else:
        print('No opportunities detected')

    # ... rest (trade confirmation block) stays the same
# Looks up and prints current positions in stock_A and stock_B
def print_positions(exchange):
    positions = exchange.get_positions()

    # Get individual instrument positions
    pos_A = positions.get(stock_A, 0)
    pos_B = positions.get(stock_B, 0)

    print('Pos(A) = ' + str(pos_A))
    print('Pos(B) = ' + str(pos_B))

    # Calculate spread position and make sure it is negative if pos_A is negative
    spread = min(abs(pos_A), abs(pos_B))
    if pos_A < 0:
        spread = -1*spread

    # Print Spread position
    print('Pos(Spread) = ' + str(spread) )

    # Calculate Spill position i.e. what is left over - this is something we need to manage!
    spill = pos_A + pos_B

    # Print spill position
    print('Pos(Spill) = ' + str(spill))

# Checks to see if we have an outright position and calls balancing_order_on_stock to fix this if required
def position_balancer(exchange):
    
    # Get positions
    positions = exchange.get_positions()
    pos_A = positions.get(stock_A, 0)
    pos_B = positions.get(stock_B, 0)
    pos_sum = pos_A + pos_B

    if(pos_sum == 0):

        print('Position is balanced. No intervention required.')

    elif(pos_sum < 0):

        # Send an order to buy some stock.
        print('pos_sum < 0: need to buy')

        # How much to trade? Whichever is smaller of abs(pos_sum) and the parameter volume_to_balance
        amount_to_trade = min(abs(pos_sum), volume_to_balance)

        # Attempt to trade. Use balancing_order_on_stock function
        balancing_order_on_stock(exchange,'bid', amount_to_trade)

    else:

        # Send an order to sell 
        print('pos_sum > 0: need to sell')

        # How much to trade? Whichever is smaller of abs(pos_sum) and the parameter volume_to_balance
        amount_to_trade = min(abs(pos_sum), volume_to_balance)

        # Attempt to trade. Use balancing_order_on_stock function
        balancing_order_on_stock(exchange,'ask', amount_to_trade)

# Here is a function for quick direct execution at the best price.
def balancing_order_on_stock(exchange, direction, amount_to_trade):

    if direction == 'bid':

        # We need to find the best ask price to trade with
        # Get the order books
        order_book_A = exchange.get_last_price_book(stock_A)
        order_book_B = exchange.get_last_price_book(stock_B)

        # Get the asks
        best_ask_A = order_book_A.asks[0].price
        best_ask_B = order_book_B.asks[0].price

        # Send a buy for requested volume into the one with the lower ask. 
        if best_ask_A <= best_ask_B:
            exchange.insert_order(stock_A, price=best_ask_A+max_tolerance_to_rebalance, volume=amount_to_trade, side='bid', order_type='ioc')
        else:
            exchange.insert_order(stock_B, price=best_ask_B+max_tolerance_to_rebalance, volume=amount_to_trade, side='bid', order_type='ioc')

    elif direction == 'ask':

        # We need to find the best bid price to trade with
        # Get the order books
        order_book_A = exchange.get_last_price_book(stock_A)
        order_book_B = exchange.get_last_price_book(stock_B)

        # Get the bids
        best_bid_A = order_book_A.bids[0].price
        best_bid_B = order_book_B.bids[0].price

        # Send a sell order for requested volume into the one with the higher bid:
        if best_bid_A >= best_bid_B:
            exchange.insert_order(stock_A, price=best_bid_A-max_tolerance_to_rebalance, volume=amount_to_trade, side='ask', order_type='ioc')
        else:
            exchange.insert_order(stock_B, price=best_bid_B-max_tolerance_to_rebalance, volume=amount_to_trade, side='ask', order_type='ioc')




def cancel_all_outstanding_orders(exchange):
    """Cancel every resting order on both instruments before switching phases."""
    exchange.delete_orders(stock_A)
    exchange.delete_orders(stock_B)
    print(">>> Cancelled all outstanding orders before switching phases.")


 
def main():
    exchange = Exchange()
    exchange.connect()
 
    init_logging()
 
    print("Starting bot for", INSTRUMENT_IDS)
    start_time = time.time()
 
    try:
        while time.time() - start_time < RUN_DURATION_SECONDS:
            # --- Phase 1: Active spread-arb trading, for LOOPS_PER_PHASE loops ---
            print("\n=== Starting ACTIVE TRADING phase ===")
            for i in range(LOOPS_PER_PHASE):
                if time.time() - start_time >= RUN_DURATION_SECONDS:
                    break
                print(f"\n[Active trading {i+1}/{LOOPS_PER_PHASE}] {datetime.datetime.now()}\n")
 
                # Pull fresh books + positions for this iteration - this is
                # what was missing before (best_bid_A etc. were never set).
                (
                    positions_initial,
                    current_position_A,
                    current_position_B,
                    order_book_A,
                    order_book_B,
                    best_bid_A,
                    best_ask_A,
                    best_bid_B,
                    best_ask_B,
                ) = positions(exchange)
 
                target_status, target_volume = stellage(
                    current_position_A=current_position_A,
                    best_bid_A=best_bid_A,
                    best_ask_A=best_ask_A,
                    best_bid_B=best_bid_B,
                    best_ask_B=best_ask_B,
                )
                trade_loop(exchange, positions_initial, target_status, target_volume, best_bid_A, best_ask_A, best_bid_B, best_ask_B)
                print_positions(exchange)
                position_balancer(exchange)
 
                # Log post-trade state for both instruments and pick up any new fills
                log_loop_state(exchange, phase='active', instrument_id=stock_A, best_bid=best_bid_A, best_ask=best_ask_A)
                log_loop_state(exchange, phase='active', instrument_id=stock_B, best_bid=best_bid_B, best_ask=best_ask_B)
                log_new_trades(exchange, stock_A, phase='active')
                log_new_trades(exchange, stock_B, phase='active')
 
                time.sleep(0.5)
 
            cancel_all_outstanding_orders(exchange)
 
            if time.time() - start_time >= RUN_DURATION_SECONDS:
                break
 
            # --- Phase 2: Passive market making, for LOOPS_PER_PHASE loops ---
            print("\n=== Starting MARKET MAKING phase ===")
            for i in range(LOOPS_PER_PHASE):
                if time.time() - start_time >= RUN_DURATION_SECONDS:
                    break
                print(f"\n[Market making {i+1}/{LOOPS_PER_PHASE}] {datetime.datetime.now()}\n")
 
                for instrument_id in INSTRUMENT_IDS:
                    quote_instrument(exchange, instrument_id)
 
                time.sleep(sleep_seconds)
 
            cancel_all_outstanding_orders(exchange)
 
    finally:
        cancel_all_outstanding_orders(exchange)
        close_logging()
        print(f"\nRun complete. Elapsed: {time.time() - start_time:.0f}s")
 
 
if __name__ == '__main__':
    main()